In [69]:
import sys
from pathlib import Path
import os

cwd = Path(os.getcwd())
repo_root = cwd

while repo_root.name != "data-pipeline-track" and repo_root.parent != repo_root:
    repo_root = repo_root.parent

notebook_dir = repo_root / "notebooks"
load_file = notebook_dir / "LoadEnvAndSetupSession.py"

if not load_file.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {load_file}")

sys.path.insert(0, str(notebook_dir))
sys.path.insert(0, str(repo_root))
sys.path.insert(0, str(repo_root / "platform" / "shared-libs"))

exec(open(load_file).read())

Exception: File `'LoadEnvAndSetupSession.py'` not found.

In [ ]:
from track_platform.logging import setup_logger
from typing import Dict
import os
import atexit
import re
import time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed
import clickhouse_connect
from track_platform.tunnel import SSHTunnelManager

logger = setup_logger("oracle_quick_test")

if 'spark' not in globals():
    logger.error("Spark não encontrado em globals()")
    raise NameError("spark não está disponível")

test_df = spark.range(1).limit(1)
test_df.collect()

logger.info(f"Spark versão: {spark.version}")
logger.info(f"App Name: {spark.sparkContext.appName}")
logger.info(f"Master: {spark.sparkContext.master}")

try:
    ui_url = spark.sparkContext.uiWebUrl
    if ui_url:
        logger.info(f"Spark UI: {ui_url}")
except:
    pass

In [ ]:
ssh_tunnel_manager = SSHTunnelManager.from_env()
ssh_tunnel = None

if ssh_tunnel_manager:
    try:
        from sshtunnel import SSHTunnelForwarder
        
        ssh_auth = {}
        if ssh_tunnel_manager.ssh_password:
            ssh_auth["ssh_password"] = ssh_tunnel_manager.ssh_password
        elif ssh_tunnel_manager.ssh_pkey:
            pkey_path = Path(ssh_tunnel_manager.ssh_pkey) if isinstance(ssh_tunnel_manager.ssh_pkey, str) else ssh_tunnel_manager.ssh_pkey
            if isinstance(pkey_path, Path) and pkey_path.exists():
                ssh_auth["ssh_pkey"] = str(pkey_path)
            else:
                ssh_auth["ssh_pkey"] = ssh_tunnel_manager.ssh_pkey
        
        ssh_tunnel_manager.tunnel = SSHTunnelForwarder(
            (ssh_tunnel_manager.ssh_host, ssh_tunnel_manager.ssh_port),
            ssh_username=ssh_tunnel_manager.ssh_user,
            remote_bind_address=(ssh_tunnel_manager.remote_host, ssh_tunnel_manager.remote_port),
            local_bind_address=("127.0.0.1", ssh_tunnel_manager.local_bind_port),
            set_keepalive=30,
            **ssh_auth,
        )
        
        ssh_tunnel_manager.tunnel.start()
        ssh_tunnel_manager.local_bind_port = ssh_tunnel_manager.tunnel.local_bind_port
        ssh_tunnel = ssh_tunnel_manager
        
        logger.info(f"Túnel SSH estabelecido: porta local {ssh_tunnel.local_bind_port}")
        
        def cleanup_tunnel():
            if ssh_tunnel and ssh_tunnel.is_active:
                ssh_tunnel.tunnel.stop()
                logger.info("Túnel SSH encerrado")
        atexit.register(cleanup_tunnel)
        
    except Exception as e:
        logger.warning(f"Falha ao iniciar túnel SSH: {e}")
        ssh_tunnel = None
else:
    logger.info("Túnel SSH não configurado")
    ssh_tunnel = None

In [ ]:
def _oracle_jdbc_config(prefix: str, tunnel=None, direct_config=None) -> Dict[str, str]:
    if direct_config:
        host = direct_config.get('host', '10.255.150.11')
        port = str(direct_config.get('port', '1521'))
        service = direct_config.get('service', 'bi.grupotracker.com.br')
        user = direct_config.get('user', 'C##AMARO_BE')
        password = direct_config.get('password', 'qi#U!E0oe123')
    elif tunnel and tunnel.is_active:
        host = "127.0.0.1"
        port = str(tunnel.local_bind_port)
        service = os.getenv(f"ORACLE_{prefix}_SERVICE", "bi.grupotracker.com.br")
        user = os.getenv(f"ORACLE_{prefix}_USER", "C##AMARO_BE")
        password = os.getenv(f"ORACLE_{prefix}_PASSWORD", "qi#U!E0oe123")
    else:
        host = os.getenv(f"ORACLE_{prefix}_HOST", "10.255.150.11")
        port = os.getenv(f"ORACLE_{prefix}_PORT", "1521")
        service = os.getenv(f"ORACLE_{prefix}_SERVICE", "bi.grupotracker.com.br")
        user = os.getenv(f"ORACLE_{prefix}_USER", "C##AMARO_BE")
        password = os.getenv(f"ORACLE_{prefix}_PASSWORD", "qi#U!E0oe123")
    
    jdbc_url = f"jdbc:oracle:thin:@//{host}:{port}/{service}"
    return {
        "url": jdbc_url,
        "properties": {
            "user": user,
            "password": password,
            "driver": "oracle.jdbc.driver.OracleDriver",
        },
    }

direct_oracle_config = {
    'host': '10.255.150.11',
    'port': '1521',
    'service': 'bi.grupotracker.com.br',
    'user': 'C##AMARO_BE',
    'password': 'qi#U!E0oe123'
}

if ssh_tunnel and ssh_tunnel.is_active:
    oracle_scot_cfg = _oracle_jdbc_config("SCOT", tunnel=ssh_tunnel)
    oracle_ginf_cfg = _oracle_jdbc_config("GINF", tunnel=ssh_tunnel)
    oracle_siga_cfg = _oracle_jdbc_config("SIGA", tunnel=ssh_tunnel)
    oracle_scot_cfg["properties"]["user"] = direct_oracle_config['user']
    oracle_scot_cfg["properties"]["password"] = direct_oracle_config['password']
    oracle_ginf_cfg["properties"]["user"] = direct_oracle_config['user']
    oracle_ginf_cfg["properties"]["password"] = direct_oracle_config['password']
    oracle_siga_cfg["properties"]["user"] = direct_oracle_config['user']
    oracle_siga_cfg["properties"]["password"] = direct_oracle_config['password']
else:
    oracle_scot_cfg = _oracle_jdbc_config("SCOT", tunnel=None, direct_config=direct_oracle_config)
    oracle_ginf_cfg = _oracle_jdbc_config("GINF", tunnel=None, direct_config=direct_oracle_config)
    oracle_siga_cfg = _oracle_jdbc_config("SIGA", tunnel=None, direct_config=direct_oracle_config)

logger.info(f"Oracle SCOT URL: {oracle_scot_cfg['url']}")
logger.info(f"Oracle GINF URL: {oracle_ginf_cfg['url']}")
logger.info(f"Oracle SIGA URL: {oracle_siga_cfg['url']}")

In [ ]:
def test_oracle_connection(cfg, schema_name: str) -> bool:
    try:
        test_query = "SELECT USER as current_user, SYSDATE as current_time FROM dual"
        test_df = spark.read.jdbc(
            cfg["url"],
            table=f"({test_query}) test_alias",
            properties=cfg["properties"]
        )
        test_results = test_df.collect()
        
        if test_results:
            test_row = test_results[0]
            user = test_row['CURRENT_USER'] if 'CURRENT_USER' in test_row.asDict() else test_row[0]
            logger.info(f"Conexão Oracle {schema_name} bem-sucedida - Usuário: {user}")
            return True
        return False
    except Exception as e:
        logger.error(f"Falha na conexão Oracle {schema_name}: {e}")
        return False

test_ginf = test_oracle_connection(oracle_ginf_cfg, "GINF")
test_scot = test_oracle_connection(oracle_scot_cfg, "SCOT")
test_siga = test_oracle_connection(oracle_siga_cfg, "SIGA")

if test_ginf:
    logger.info("Listando primeiras 5 tabelas do schema GINF...")
    try:
        tables_query = "(SELECT table_name FROM all_tables WHERE owner = 'GINF' ORDER BY table_name) tables_alias"
        df_tables_ginf = spark.read.jdbc(
            oracle_ginf_cfg["url"],
            table=tables_query,
            properties=oracle_ginf_cfg["properties"]
        ).limit(5)
        
        tables_list = df_tables_ginf.collect()
        logger.info(f"Encontradas {len(tables_list)} tabelas (mostrando primeiras 5):")
        for row in tables_list:
            table_name = row['TABLE_NAME'] if 'TABLE_NAME' in row.asDict() else row[0]
            logger.info(f"  - {table_name}")
            
            sample_query = f"(SELECT * FROM GINF.{table_name} WHERE ROWNUM <= 3) sample_alias"
            try:
                df_sample = spark.read.jdbc(
                    oracle_ginf_cfg["url"],
                    table=sample_query,
                    properties=oracle_ginf_cfg["properties"]
                )
                logger.info(f"    Colunas: {', '.join(df_sample.columns[:5])}... ({len(df_sample.columns)} total)")
                logger.info(f"    Linhas de amostra: {df_sample.count()}")
            except Exception as e:
                logger.warning(f"    Não foi possível ler amostra da tabela {table_name}: {e}")
                
    except Exception as e:
        logger.error(f"Erro ao listar tabelas GINF: {e}")

if test_scot:
    logger.info("Verificando schemas disponíveis e listando tabelas SCOT...")
    try:
        schemas_to_try = ['SCOT_OWNER', 'SCOT', 'SCOTT']
        
        for schema_name in schemas_to_try:
            logger.info(f"Tentando schema: {schema_name}")
            tables_query = f"(SELECT table_name FROM all_tables WHERE owner = '{schema_name}' ORDER BY table_name) tables_alias"
            
            try:
                df_tables_scot = spark.read.jdbc(
                    oracle_scot_cfg["url"],
                    table=tables_query,
                    properties=oracle_scot_cfg["properties"]
                ).limit(5)
                
                tables_list = df_tables_scot.collect()
                
                if len(tables_list) > 0:
                    logger.info(f"Schema encontrado: {schema_name} - {len(tables_list)} tabelas (mostrando primeiras 5):")
                    for row in tables_list:
                        table_name = row['TABLE_NAME'] if 'TABLE_NAME' in row.asDict() else row[0]
                        logger.info(f"  - {table_name}")
                        
                        sample_query = f"(SELECT * FROM {schema_name}.{table_name} WHERE ROWNUM <= 3) sample_alias"
                        try:
                            df_sample = spark.read.jdbc(
                                oracle_scot_cfg["url"],
                                table=sample_query,
                                properties=oracle_scot_cfg["properties"]
                            )
                            logger.info(f"    Colunas: {', '.join(df_sample.columns[:5])}... ({len(df_sample.columns)} total)")
                            logger.info(f"    Linhas de amostra: {df_sample.count()}")
                        except Exception as e:
                            logger.warning(f"    Não foi possível ler amostra da tabela {table_name}: {e}")
                    break
                else:
                    logger.debug(f"Schema {schema_name} não retornou tabelas")
            except Exception as e:
                logger.debug(f"Erro ao consultar schema {schema_name}: {e}")
                continue
        else:
            logger.warning("Nenhum schema SCOT encontrado. Verificando todos os schemas disponíveis...")
            all_schemas_query = "(SELECT DISTINCT owner FROM all_tables WHERE owner LIKE '%SCOT%' ORDER BY owner) schemas_alias"
            try:
                df_schemas = spark.read.jdbc(
                    oracle_scot_cfg["url"],
                    table=all_schemas_query,
                    properties=oracle_scot_cfg["properties"]
                )
                schemas_list = df_schemas.collect()
                if schemas_list:
                    logger.info("Schemas encontrados com 'SCOT' no nome:")
                    for row in schemas_list:
                        schema = row['OWNER'] if 'OWNER' in row.asDict() else row[0]
                        logger.info(f"  - {schema}")
                else:
                    logger.warning("Nenhum schema com 'SCOT' no nome encontrado")
            except Exception as e:
                logger.error(f"Erro ao listar schemas: {e}")
                
    except Exception as e:
        logger.error(f"Erro ao listar tabelas SCOT: {e}")

if test_siga:
    logger.info("Listando primeiras 5 tabelas do schema SIGA...")
    try:
        schemas_to_try = ['SIGA', 'SIGA_OWNER']
        
        for schema_name in schemas_to_try:
            logger.info(f"Tentando schema: {schema_name}")
            tables_query = f"(SELECT table_name FROM all_tables WHERE owner = '{schema_name}' ORDER BY table_name) tables_alias"
            
            try:
                df_tables_siga = spark.read.jdbc(
                    oracle_siga_cfg["url"],
                    table=tables_query,
                    properties=oracle_siga_cfg["properties"]
                ).limit(5)
                
                tables_list = df_tables_siga.collect()
                
                if len(tables_list) > 0:
                    logger.info(f"Schema encontrado: {schema_name} - {len(tables_list)} tabelas (mostrando primeiras 5):")
                    for row in tables_list:
                        table_name = row['TABLE_NAME'] if 'TABLE_NAME' in row.asDict() else row[0]
                        logger.info(f"  - {table_name}")
                        
                        sample_query = f"(SELECT * FROM {schema_name}.{table_name} WHERE ROWNUM <= 3) sample_alias"
                        try:
                            df_sample = spark.read.jdbc(
                                oracle_siga_cfg["url"],
                                table=sample_query,
                                properties=oracle_siga_cfg["properties"]
                            )
                            logger.info(f"    Colunas: {', '.join(df_sample.columns[:5])}... ({len(df_sample.columns)} total)")
                            logger.info(f"    Linhas de amostra: {df_sample.count()}")
                        except Exception as e:
                            logger.warning(f"    Não foi possível ler amostra da tabela {table_name}: {e}")
                    break
                else:
                    logger.debug(f"Schema {schema_name} não retornou tabelas")
            except Exception as e:
                logger.debug(f"Erro ao consultar schema {schema_name}: {e}")
                continue
        else:
            logger.warning("Nenhum schema SIGA encontrado. Verificando todos os schemas disponíveis...")
            all_schemas_query = "(SELECT DISTINCT owner FROM all_tables WHERE owner LIKE '%SIGA%' ORDER BY owner) schemas_alias"
            try:
                df_schemas = spark.read.jdbc(
                    oracle_siga_cfg["url"],
                    table=all_schemas_query,
                    properties=oracle_siga_cfg["properties"]
                )
                schemas_list = df_schemas.collect()
                if schemas_list:
                    logger.info("Schemas encontrados com 'SIGA' no nome:")
                    for row in schemas_list:
                        schema = row['OWNER'] if 'OWNER' in row.asDict() else row[0]
                        logger.info(f"  - {schema}")
                else:
                    logger.warning("Nenhum schema com 'SIGA' no nome encontrado")
            except Exception as e:
                logger.error(f"Erro ao listar schemas: {e}")
                
    except Exception as e:
        logger.error(f"Erro ao listar tabelas SIGA: {e}")

In [ ]:
def _process_single_table(table_name: str, owner_up: str, cfg: Dict[str, str], ddl_dir, idx: int, total: int) -> dict:
    try:
        logger.debug(f"Processando tabela [{idx}/{total}]: {table_name}")
        
        columns_query = f"""
            (SELECT 
                column_name,
                data_type,
                data_length,
                data_precision,
                data_scale,
                nullable
             FROM all_tab_columns 
             WHERE owner = '{owner_up}' 
               AND table_name = '{table_name}'
             ORDER BY column_id) columns_alias
        """
        
        df_columns = spark.read.jdbc(
            cfg["url"], 
            table=columns_query, 
            properties=cfg["properties"]
        )
        
        columns_list = df_columns.collect()
        
        if not columns_list:
            logger.warning(f"Nenhuma coluna encontrada para {table_name}")
            return None
        
        clickhouse_columns = []
        for col_row in columns_list:
            col_name = None
            if hasattr(col_row, 'COLUMN_NAME'):
                col_name = col_row.COLUMN_NAME
            elif 'COLUMN_NAME' in col_row.asDict():
                col_name = col_row.asDict()['COLUMN_NAME']
            elif len(col_row) > 0:
                col_name = col_row[0]
            
            if not col_name:
                continue
            
            data_type = None
            if hasattr(col_row, 'DATA_TYPE'):
                data_type = col_row.DATA_TYPE
            elif 'DATA_TYPE' in col_row.asDict():
                data_type = col_row.asDict()['DATA_TYPE']
            elif len(col_row) > 1:
                data_type = col_row[1]
            
            if not data_type:
                continue
            
            type_mapping = {
                'NUMBER': 'Decimal64(2)',
                'VARCHAR2': 'String',
                'CHAR': 'String',
                'DATE': 'DateTime',
                'TIMESTAMP': 'DateTime64(3)',
                'CLOB': 'String',
                'BLOB': 'String',
            }
            
            ch_type = type_mapping.get(data_type.upper(), 'String')
            col_name_clean = col_name.lower().replace(' ', '_').replace('-', '_').replace('ç', 'c').replace('é', 'e').replace('ó', 'o').replace('º', 'o').replace('á', 'a').replace('í', 'i').replace('ú', 'u')
            clickhouse_columns.append(f"    {col_name_clean} {ch_type}")
        
        if not clickhouse_columns:
            logger.warning(f"Não foi possível gerar DDL para {table_name}")
            return None
        
        newline = '\n'
        columns_str = f',{newline}'.join(clickhouse_columns)
        ch_table_name = f"{owner_up.lower()}_{table_name.lower()}"
        ddl = f"""CREATE TABLE IF NOT EXISTS tracker_bronze.{ch_table_name}
(
{columns_str},
    _ingestion_timestamp DateTime DEFAULT now(),
    reference_date Date,
    _source String,
    _source_table String,
    row_hash String
)
ENGINE = MergeTree()
PARTITION BY toYYYYMM(reference_date)
ORDER BY (reference_date, row_hash)
SETTINGS index_granularity = 8192;
"""
        
        if ddl_dir:
            try:
                ddl_file = ddl_dir / f"bronze_{ch_table_name}.sql"
                ddl_file.write_text(ddl, encoding='utf-8')
                logger.info(f"DDL salvo: {table_name} -> {ch_table_name} ({len(columns_list)} colunas)")
                return {
                    'file': str(ddl_file),
                    'ddl': ddl,
                    'table': ch_table_name,
                    'oracle_table': f"{owner_up}.{table_name}"
                }
            except Exception as e:
                logger.error(f"Erro ao salvar DDL para {table_name}: {e}")
                return None
        
        return {
            'file': None,
            'ddl': ddl,
            'table': ch_table_name,
            'oracle_table': f"{owner_up}.{table_name}"
        }
        
    except Exception as e:
        logger.error(f"Erro ao processar tabela {table_name}: {e}")
        return None


def generate_bronze_ddls(owner: str, cfg: Dict[str, str], limit: int = None, save_to_file: bool = True, max_workers: int = 8) -> list:
    try:
        if 'spark' not in globals():
            logger.error("Spark não está disponível")
            return []
        
        ddl_dir = None
        if save_to_file:
            repo_root = Path.cwd()
            while repo_root.name != "data-pipeline-track" and repo_root.parent != repo_root:
                repo_root = repo_root.parent
            ddl_dir = repo_root / "domains" / "data-pipeline" / "clickhouse" / "ddl" / "tracker_bronze"
            ddl_dir.mkdir(parents=True, exist_ok=True)
            logger.info(f"DDLs serão salvos em: {ddl_dir}")
        
        owner_up = owner.upper()
        base_query = f"(SELECT table_name FROM all_tables WHERE owner = '{owner_up}' ORDER BY table_name) tables_alias"
        
        df_tables = spark.read.jdbc(cfg["url"], table=base_query, properties=cfg["properties"]).orderBy("table_name")
        
        if limit:
            df_tables = df_tables.limit(limit)
        
        tables_list = [row['TABLE_NAME'] if 'TABLE_NAME' in row.asDict() else row[0] for row in df_tables.collect()]
        total_tables = len(tables_list)
        
        logger.info(f"Gerando DDLs para {total_tables} tabelas do schema {owner_up}")
        
        ddls_gerados = []
        start_time = time.time()
        
        from concurrent.futures import ThreadPoolExecutor, as_completed
        
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = {
                executor.submit(_process_single_table, table_name, owner_up, cfg, ddl_dir, idx + 1, total_tables): table_name
                for idx, table_name in enumerate(tables_list)
            }
            
            for future in as_completed(futures):
                table_name = futures[future]
                try:
                    result = future.result()
                    if result:
                        ddls_gerados.append(result)
                except Exception as e:
                    logger.error(f"Erro ao processar {table_name}: {e}")
        
        elapsed_time = time.time() - start_time
        logger.info(f"DDLs gerados: {len(ddls_gerados)}/{total_tables} em {elapsed_time:.2f}s")
        
        return ddls_gerados
        
    except Exception as e:
        logger.error(f"Falha ao gerar DDLs: {e}")
        import traceback
        traceback.print_exc()
        return []

In [ ]:
def _execute_single_ddl(sql_file, idx: int, total: int, create_client_func) -> dict:
    try:
        ddl = sql_file.read_text(encoding='utf-8')
        
        match = re.search(r'CREATE TABLE IF NOT EXISTS tracker_bronze\.(\w+)', ddl, re.IGNORECASE)
        table_name = match.group(1) if match else sql_file.stem
        
        clickhouse_client = create_client_func()
        clickhouse_client.command(ddl)
        clickhouse_client.close()
        
        logger.info(f"Tabela criada [{idx}/{total}]: tracker_bronze.{table_name}")
        
        return {
            'file': sql_file.name,
            'table': table_name,
            'status': 'success',
            'idx': idx
        }
        
    except Exception as e:
        logger.error(f"Erro ao executar {sql_file.name} [{idx}/{total}]: {e}")
        return {
            'file': sql_file.name,
            'table': table_name if 'table_name' in locals() else 'unknown',
            'status': 'error',
            'error': str(e),
            'idx': idx
        }


def execute_bronze_ddls(ddl_dir_path: str = None, max_workers: int = 8) -> None:
    try:
        def create_clickhouse_client():
            host = os.getenv('CLICKHOUSE_HOST') or os.getenv('CLICKHOUSE_ENDPOINT')
            if not host:
                raise RuntimeError("CLICKHOUSE_HOST não definido")
            
            port = int(os.getenv('CLICKHOUSE_PORT', '8443'))
            username = os.getenv('CLICKHOUSE_USER', 'default')
            password = os.getenv('CLICKHOUSE_PASSWORD')
            if not password:
                raise RuntimeError("CLICKHOUSE_PASSWORD não definido")
            
            database = os.getenv('CLICKHOUSE_DATABASE') or os.getenv('CLICKHOUSE_DB') or 'default'
            secure = str(os.getenv('CLICKHOUSE_SECURE', 'true')).lower() in ('1', 'true', 'yes')
            verify = str(os.getenv('CLICKHOUSE_VERIFY', 'true')).lower() in ('1', 'true', 'yes')
            
            if host.endswith('.clickhouse.cloud'):
                if port != 8443:
                    raise RuntimeError("ClickHouse Cloud requer porta 8443")
                if not secure:
                    raise RuntimeError("ClickHouse Cloud requer TLS habilitado")
            
            return clickhouse_connect.get_client(
                host=host,
                port=port,
                username=username,
                password=password,
                database=database,
                secure=secure,
                verify=verify,
                connect_timeout=10,
                send_receive_timeout=60,
                client_name='track-ddl-deployer'
            )
        
        if ddl_dir_path is None:
            repo_root = Path.cwd()
            while repo_root.name != "data-pipeline-track" and repo_root.parent != repo_root:
                repo_root = repo_root.parent
            ddl_dir = repo_root / "domains" / "data-pipeline" / "clickhouse" / "ddl" / "tracker_bronze"
        else:
            ddl_dir = Path(ddl_dir_path)
        
        if not ddl_dir.exists():
            logger.error(f"Diretório não encontrado: {ddl_dir}")
            return
        
        clickhouse_client = create_clickhouse_client()
        clickhouse_client.command("CREATE DATABASE IF NOT EXISTS tracker_bronze")
        clickhouse_client.close()
        
        sql_files = sorted(ddl_dir.glob("*.sql"))
        
        if not sql_files:
            logger.warning(f"Nenhum arquivo DDL encontrado em: {ddl_dir}")
            return
        
        total_files = len(sql_files)
        logger.info(f"Executando {total_files} DDLs no ClickHouse (paralelo, {max_workers} workers)")
        
        sucesso = 0
        falhas = 0
        start_time = time.time()
        
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = {
                executor.submit(_execute_single_ddl, sql_file, idx + 1, total_files, create_clickhouse_client): sql_file
                for idx, sql_file in enumerate(sql_files)
            }
            
            for future in as_completed(futures):
                sql_file = futures[future]
                try:
                    result = future.result()
                    if result and result.get('status') == 'success':
                        sucesso += 1
                    else:
                        falhas += 1
                except Exception as e:
                    logger.error(f"Erro ao processar {sql_file.name}: {e}")
                    falhas += 1
        
        elapsed_time = time.time() - start_time
        logger.info(f"Execução concluída: {sucesso} sucesso, {falhas} falhas em {elapsed_time:.2f}s")
        
    except Exception as e:
        logger.error(f"Falha ao executar DDLs: {e}")
        import traceback
        traceback.print_exc()

In [ ]:
if 'generate_bronze_ddls' not in globals():
    raise NameError("Execute a célula 'generate_ddls' primeiro para definir as funções")

ddls_ginf = generate_bronze_ddls("GINF", oracle_ginf_cfg, limit=None, max_workers=8)

In [ ]:
if 'generate_bronze_ddls' not in globals():
    raise NameError("Execute a célula 'generate_ddls' primeiro para definir as funções")

ddls_scot = generate_bronze_ddls("SCOT_OWNER", oracle_scot_cfg, limit=None, max_workers=8)

In [40]:
if 'generate_bronze_ddls' not in globals():
    raise NameError("Execute a célula 'generate_ddls' primeiro para definir as funções")

ddls_siga = generate_bronze_ddls("SIGA", oracle_siga_cfg, limit=None, max_workers=8)

NameError: name 'oracle_siga_cfg' is not defined

In [ ]:
from pathlib import Path
import re

def verify_siga_ddls():
    logger.info("Verificando DDLs do SIGA...")
    
    ddl_dir = Path(repo_root) / "domains" / "data-pipeline" / "clickhouse" / "ddl" / "tracker_bronze"
    
    if not ddl_dir.exists():
        logger.error(f"Diretório de DDLs não encontrado: {ddl_dir}")
        return
    
    logger.info("Listando todas as tabelas do schema SIGA no Oracle...")
    try:
        schemas_to_try = ['SIGA', 'SIGA_OWNER']
        oracle_tables = set()
        schema_found = None
        
        for schema_name in schemas_to_try:
            try:
                tables_query = f"(SELECT table_name FROM all_tables WHERE owner = '{schema_name}' ORDER BY table_name) tables_alias"
                df_tables = spark.read.jdbc(
                    oracle_siga_cfg["url"],
                    table=tables_query,
                    properties=oracle_siga_cfg["properties"]
                )
                
                tables_list = df_tables.collect()
                
                if len(tables_list) > 0:
                    schema_found = schema_name
                    for row in tables_list:
                        table_name = row['TABLE_NAME'] if 'TABLE_NAME' in row.asDict() else row[0]
                        oracle_tables.add(table_name.upper())
                    break
            except Exception as e:
                logger.debug(f"Erro ao consultar schema {schema_name}: {e}")
                continue
        
        if not schema_found:
            logger.error("Nenhum schema SIGA encontrado no Oracle")
            return
        
        logger.info(f"Schema encontrado: {schema_found}")
        logger.info(f"Total de tabelas no Oracle: {len(oracle_tables)}")
        
        logger.info("Listando DDLs já gerados...")
        ddl_files = list(ddl_dir.glob("bronze_siga_*.sql"))
        ddl_tables = set()
        
        for ddl_file in ddl_files:
            filename = ddl_file.stem
            match = re.match(r'^bronze_siga_(.+)$', filename)
            if match:
                table_name = match.group(1).upper()
                ddl_tables.add(table_name)
        
        logger.info(f"Total de DDLs gerados: {len(ddl_tables)}")
        
        missing_tables = oracle_tables - ddl_tables
        extra_ddls = ddl_tables - oracle_tables
        
        logger.info("=" * 80)
        logger.info("RESUMO DA VERIFICAÇÃO SIGA")
        logger.info("=" * 80)
        logger.info(f"Tabelas no Oracle: {len(oracle_tables)}")
        logger.info(f"DDLs gerados: {len(ddl_tables)}")
        logger.info(f"Tabelas faltando DDL: {len(missing_tables)}")
        logger.info(f"DDLs extras (tabelas não encontradas no Oracle): {len(extra_ddls)}")
        
        if missing_tables:
            logger.warning(f"\nTabelas sem DDL ({len(missing_tables)}):")
            for table in sorted(missing_tables):
                logger.warning(f"  - {table}")
        else:
            logger.info("\n✓ Todas as tabelas do SIGA têm DDLs gerados!")
        
        if extra_ddls:
            logger.warning(f"\nDDLs extras (tabelas não encontradas no Oracle) ({len(extra_ddls)}):")
            for table in sorted(list(extra_ddls)[:20]):
                logger.warning(f"  - {table}")
            if len(extra_ddls) > 20:
                logger.warning(f"  ... e mais {len(extra_ddls) - 20} tabelas")
        
        logger.info("=" * 80)
        
        return {
            'oracle_tables': oracle_tables,
            'ddl_tables': ddl_tables,
            'missing_tables': missing_tables,
            'extra_ddls': extra_ddls,
            'schema_found': schema_found
        }
        
    except Exception as e:
        logger.error(f"Erro ao verificar DDLs do SIGA: {e}")
        import traceback
        traceback.print_exc()
        return None

if 'oracle_siga_cfg' not in globals():
    logger.error("Execute a célula 'oracle_config' primeiro para definir oracle_siga_cfg")
else:
    siga_verification = verify_siga_ddls()

In [ ]:
def analyze_failed_ddls(schema_name: str, cfg: Dict[str, str], ddl_dir: Path):
    logger.info(f"Analisando DDLs falhados do schema {schema_name}...")
    
    try:
        tables_query = f"(SELECT table_name FROM all_tables WHERE owner = '{schema_name}' ORDER BY table_name) tables_alias"
        df_tables = spark.read.jdbc(
            cfg["url"],
            table=tables_query,
            properties=cfg["properties"]
        )
        
        oracle_tables = set()
        for row in df_tables.collect():
            table_name = row['TABLE_NAME'] if 'TABLE_NAME' in row.asDict() else row[0]
            oracle_tables.add(table_name.upper())
        
        logger.info(f"Total de tabelas no Oracle ({schema_name}): {len(oracle_tables)}")
        
        ddl_files = list(ddl_dir.glob(f"bronze_{schema_name.lower()}_*.sql"))
        ddl_tables = set()
        
        for ddl_file in ddl_files:
            filename = ddl_file.stem
            match = re.match(rf'^bronze_{schema_name.lower()}_(.+)$', filename)
            if match:
                table_name = match.group(1).upper()
                ddl_tables.add(table_name)
        
        logger.info(f"Total de DDLs gerados: {len(ddl_tables)}")
        
        missing_tables = oracle_tables - ddl_tables
        
        if missing_tables:
            logger.warning(f"Tabelas sem DDL ({len(missing_tables)}):")
            for table in sorted(missing_tables):
                logger.warning(f"  - {table}")
            
            logger.info(f"\nGerando DDLs apenas para as {len(missing_tables)} tabelas faltantes...")
            return sorted(missing_tables)
        else:
            logger.info("✓ Todas as tabelas têm DDLs gerados!")
            return []
            
    except Exception as e:
        logger.error(f"Erro ao analisar DDLs falhados: {e}")
        import traceback
        traceback.print_exc()
        return []

def retry_failed_ddls(schema_name: str, cfg: Dict[str, str], failed_tables: list, max_workers: int = 8):
    if not failed_tables:
        logger.info("Nenhuma tabela para reprocessar")
        return []
    
    logger.info(f"Reprocessando {len(failed_tables)} tabelas do schema {schema_name}...")
    
    ddl_dir = Path(repo_root) / "domains" / "data-pipeline" / "clickhouse" / "ddl" / "tracker_bronze"
    ddl_dir.mkdir(parents=True, exist_ok=True)
    
    from concurrent.futures import ThreadPoolExecutor, as_completed
    
    results = []
    total = len(failed_tables)
    
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        futures = {
            executor.submit(_process_single_table, table, schema_name, cfg, ddl_dir, idx + 1, total): table
            for idx, table in enumerate(failed_tables)
        }
        
        for future in as_completed(futures):
            table = futures[future]
            try:
                result = future.result()
                if result:
                    results.append(result)
                    logger.info(f"[{len(results)}/{total}] ✓ DDL gerado: {table}")
                else:
                    logger.warning(f"[{len(results)}/{total}] ✗ Falha ao gerar DDL: {table}")
            except Exception as e:
                logger.error(f"Erro ao processar {table}: {e}")
    
    logger.info(f"Reprocessamento concluído: {len(results)}/{total} DDLs gerados com sucesso")
    return results

if 'oracle_siga_cfg' not in globals():
    logger.error("Execute a célula 'oracle_config' primeiro")
else:
    ddl_dir = Path(repo_root) / "domains" / "data-pipeline" / "clickhouse" / "ddl" / "tracker_bronze"
    failed_siga_tables = analyze_failed_ddls("SIGA", oracle_siga_cfg, ddl_dir)
    
    if failed_siga_tables:
        logger.info(f"\nEncontradas {len(failed_siga_tables)} tabelas sem DDL")
        logger.info("Execute a célula novamente com retry_failed_ddls() para reprocessar")

In [ ]:
if 'failed_siga_tables' in globals() and failed_siga_tables:
    logger.info(f"Reprocessando {len(failed_siga_tables)} tabelas do SIGA que falharam...")
    retry_results = retry_failed_ddls("SIGA", oracle_siga_cfg, failed_siga_tables, max_workers=8)
    logger.info(f"Reprocessamento concluído: {len(retry_results)} DDLs gerados")
else:
    logger.info("Nenhuma tabela para reprocessar. Execute a célula anterior primeiro.")

In [ ]:
if 'execute_bronze_ddls' not in globals():
    raise NameError("Execute a célula 'execute_ddls' primeiro para definir as funções")

execute_bronze_ddls(max_workers=8)